# 🏁 Lab 08 — โปรเจ็คจบคอร์ส: ทำนายลูกค้าที่จะยกเลิก (Customer Churn)

**Machine Learning with scikit-learn — Session 08 · 40% ของคะแนนวิชา**

## 📋 โจทย์ธุรกิจ
บริษัทโทรคมนาคมอยากรู้ล่วงหน้าว่าลูกค้าคนไหนจะ**ยกเลิกบริการ** เพื่อส่งทีม retention ไปช่วยได้ทัน
คุณมีข้อมูลลูกค้า 7,043 คน × 19 คอลัมน์ — target คือ `Churn` (26.5% Yes)

## ✅ สิ่งที่ต้องส่ง (rubric เต็ม ๆ ในสไลด์คาบ 8)
1. **Notebook นี้** ที่ทำครบทุก section (มีคำอธิบาย markdown ทุกขั้น)
2. **ไฟล์โมเดล** `churn_pipeline.joblib` (pipeline เต็ม ไม่ใช่แค่โมเดล)
3. **สไลด์ 4-6 หน้า** present 5 นาที ตามโครงในสไลด์คาบ 8

## 🗺️ โครงการทำงาน = คอร์สทั้งหมดย่อยเป็น 8 ขั้น
ทำตามลำดับ ห้ามข้าม Step 0 (กัน leakage ตั้งแต่วินาทีแรก)

> 💡 **AI ช่วยได้ทุกขั้น** — แต่ต้องอธิบายได้ทุกบรรทัดที่ส่ง และแนบ prompt ที่ใช้ไว้ท้าย notebook

## Step 0 — Setup + โหลดข้อมูล (จับกับดัก dtype ของ TotalCharges)

In [ ]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(URL)
    print("โหลดจาก URL สำเร็จ:", df.shape)
except Exception as e:
    print("โหลดไม่ได้:", e)
    print("→ ใช้ fallback: generate synthetic churn data สำหรับฝึก")
    from sklearn.datasets import make_classification
    Xs_, ys_ = make_classification(n_samples=3000, n_features=15, n_informative=8,
                                   weights=[0.735, 0.265], random_state=42)
    df = pd.DataFrame(Xs_, columns=[f"feat_{i}" for i in range(15)])
    df["tenure"] = np.abs(Xs_[:, 0] * 20 + 30).round(0)
    df["MonthlyCharges"] = np.abs(Xs_[:, 1] * 30 + 70).round(2)
    df["TotalCharges"] = df["tenure"] * df["MonthlyCharges"]
    df["Contract"] = np.where(Xs_[:, 2] > 0, "Month-to-month", "Two year")
    df["InternetService"] = np.where(Xs_[:, 3] > 0.5, "Fiber optic", "DSL")
    df["Churn"] = np.where(ys_ == 1, "Yes", "No")
    print("synthetic:", df.shape)

# กับดัก: TotalCharges เป็น text และมีช่องว่าง ' ' ซ่อนอยู่ 11 จุด
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.head(3)

## Step 1 — EDA: พาดหัวข่าว 3 ข้อ

สำรวจแล้วเขียน **3 พาดหัวข่าว** (markdown) ที่ผู้บริหารอ่านแล้วเข้าใจทันที
แนวทาง: churn rate รวม · แยกตาม Contract / InternetService · tenure & MonthlyCharges ของคนที่ยกเลิก vs อยู่ต่อ

In [ ]:
# TODO: EDA + 3 พาดหัวข่าว

## Step 2 — แบ่งข้อมูลก่อน + Pipeline กันโปร่งใส

**แบ่ง 80:20 (stratify=y, random_state=42) ก่อนทำอย่างอื่น** แล้วสร้าง ColumnTransformer + Pipeline
(num: median impute + scale | cat: most_frequent + one-hot) — อย่าลืม drop `customerID`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# TODO: เตรียม X, y แล้ว split ก่อน + สร้าง preprocessor

## Step 3 — Baseline + เลือก metric ให้เหมาะกับธุรกิจ

Dummy ก่อนเสมอ — แล้ว**เขียนว่าคุณเลือก metric อะไรเป็นตัวตัดสิน เพราะอะไร**
(คำใบ้: 26.5% positive + บริบท retention ทำให้ accuracy ไม่ใช่คำตอบ)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# TODO: baseline + เหตุผลการเลือก metric

## Step 4 — Model zoo: เทียบอย่างน้อย 3 โมเดล (CV, ยังไม่แตะ test)

แนะนำ: LogisticRegression · RandomForest · HistGradientBoosting (อยากท้าใจ: XGBoost/LightGBM ได้ bonus)
ใช้ StratifiedKFold เดียวกัน วัดทั้ง accuracy และ **roc_auc**

In [ ]:
# TODO: เทียบ >= 3 โมเดลด้วย CV เดียวกัน

## Step 5 — Tune โมเดลที่ดีที่สุด (RandomizedSearchCV บน train เท่านั้น)

n_iter=20-30, cv=5 — แล้ว fit final บน train ทั้งหมด

In [ ]:
# TODO: RandomizedSearchCV + final fit

## Step 6 — Threshold เป็นการตัดสินใจเชิงธุรกิจ

ลอง threshold 0.35 / 0.50 / 0.65 — เขียนประโยค "decision-ready" 1 ประโยค
(เช่น "งบพอโทร 100 คน → เรียงลำดับด้วย predict_proba แล้วเลือก top 100")

In [ ]:
# TODO: threshold analysis + ประโยค decision-ready

## Step 7 — Error analysis + สิ่งที่ขับเคลื่อน churn

ดู top drivers (สัมประสิทธิ์ หรือ feature importances) + ส่อง top misses
แล้วเขียน **insight ที่ทีม retention ใช้ทำงานได้จริง** (เช่น "โทรลูกค้า fiber รายเดือนอายุ 1-6 เดือนในเดือนที่ 3")

In [ ]:
# TODO: drivers + misses + insight

## Step 8 — คำพิพากษาสุดท้าย (test ครั้งเดียว) + ส่งมอบ

วัด test set ครั้งเดียว: AUC, recall, f1, accuracy — เทียบกับ CV แล้ว**เขียนว่า gap บอกอะไร**
จากนั้น dump pipeline เป็น `churn_pipeline.joblib`

In [ ]:
import joblib
from sklearn.metrics import roc_auc_score, recall_score, f1_score

# TODO: final test (ครั้งเดียว!) + joblib.dump

## 🎤 สไลด์ 5 นาที (template จากคาบ 8)

0:00 Problem · 0:30 Data & method (1 กราฟ EDA + pipeline) · 1:30 Results (CV vs test + baseline)
· 2:30 Insight (drivers + ประโยค decision-ready) · 3:30 Limits & next · 4:30 Ask

### ✍️ เช็กลิสต์ก่อนส่ง
- [ ] รัน Runtime → Run all ได้ตั้งแต่ต้นจนจุดไม่มี error
- [ ] ทุกการตัดสินใจมี markdown อธิบายเหตุผล
- [ ] มี Dummy baseline ตลอด และเทียบ test แค่ครั้งเดียว
- [ ] joblib โหลดกลับมา + predict แถวใหม่ได้
- [ ] สไลด์ไม่มี code screenshot เลยแม้แต่ภาพเดียว

## 🤖 ลองถาม AI (vibe coding corner — ใช้เป็น code reviewer)

> "ช่วยรีวิวโค้ดโปรเจ็ค churn ของฉันเหมือน senior data scientist:
> หา leakage, metric ที่ผิดบริบท, จุดที่ควรทำ error analysis เพิ่ม และข้อจำกัดของโมเดลที่ควรพูดตอน present"

แนบ prompt และคำตอบที่มีประโยชน์ไว้ท้าย notebook ด้วย — ความโปร่งใสได้คะแนน